In [1]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import precision_recall_fscore_support

In [ ]:
dataset = load_dataset("coastalcph/tydi_xor_rc")

df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()

#filter data 
langlst = ['ko','ar','te']

df_train_filtered = df_train[df_train['lang'].isin(langlst)]
df_validation_filtered = df_validation[df_validation['lang'].isin(langlst)]

#filter data 
#Training data for each language
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_te = df_train[(df_train['lang'] == "te")]

#Validation data for each language
df_validation_ar = df_validation[(df_validation['lang'] == "ar")]
df_validation_ko = df_validation[(df_validation['lang'] == "ko")]
df_validation_te = df_validation[(df_validation['lang'] == "te")]

# df_train_filtered


: 

## Fine-tune one multilingual Transformer QA model

In [ ]:
from datasets import Dataset
from transformers import AutoModelForQuestionAnswering, Trainer, TrainingArguments

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased", use_fast=True
)
torch.manual_seed(42)

train_examples = Dataset.from_pandas(df_train_filtered, preserve_index=False)

def prepare_examples(examples):
    tokens = tokenizer(
        examples["question"], examples["context"],
        truncation="only_second", max_length=384, stride=128,
        return_overflowing_tokens=True, return_offsets_mapping=True,
        padding="max_length",
    )
    example_numbers = tokens.pop("overflow_to_sample_mapping")
    offsets = tokens.pop("offset_mapping")
    start_positions = []
    end_positions = []

    for i, example_number in enumerate(example_numbers):
        cls_position = tokens["input_ids"][i].index(tokenizer.cls_token_id)
        start_position = cls_position
        end_position = cls_position

        if examples["answerable"][example_number]:
            answer_start = int(examples["answer_start"][example_number])
            answer_end = answer_start + len(examples["answer"][example_number])
            context_tokens = [
                j for j, part in enumerate(tokens.sequence_ids(i)) if part == 1
            ]
            if (context_tokens
                and offsets[i][context_tokens[0]][0] <= answer_start
                and offsets[i][context_tokens[-1]][1] >= answer_end):
                answer_tokens = [
                    j for j in context_tokens
                    if offsets[i][j][0] < answer_end
                    and offsets[i][j][1] > answer_start
                ]
                if answer_tokens:
                    start_position = answer_tokens[0]
                    end_position = answer_tokens[-1]

        start_positions.append(start_position)
        end_positions.append(end_position)

    tokens["start_positions"] = start_positions
    tokens["end_positions"] = end_positions
    return tokens

train_features = train_examples.map(
    prepare_examples, batched=True,
    remove_columns=train_examples.column_names,
)

model = AutoModelForQuestionAnswering.from_pretrained(
    "bert-base-multilingual-cased"
)
training_args = TrainingArguments(
    output_dir="multilingual-qa", num_train_epochs=1,
    per_device_train_batch_size=8, learning_rate=3e-5,
    seed=42, save_strategy="no", report_to="none",
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_features)
trainer.train()

Map:   0%|          | 0/6335 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w